In [1]:
import sys
import os
import datetime
import copy
from multiprocessing.dummy import Pool
import pickle

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
plt.rc('font',family='Times New Roman')

__file__ = os.getcwd()
project_path = os.path.dirname(os.path.abspath(__file__))
sys.path.append(project_path)

from simulation_env.environment import *

In [2]:
def draw_histogram(y1, y2, y3):
    x = np.arange(10) #总共有几组，就设置成几，我们这里有三组，所以设置为3
    total_width, n = 0.8, 3    # 有多少个类型，只需更改n即可，比如这里我们对比了四个，那么就把n设成4
    width = total_width / n
    x = x - (total_width - width) / 2
    plt.bar(x, y1, color = "maroon",width=width,label='Random policy ')
    plt.bar(x + width, y2, color = "darkorange",width=width,label='MADQN ')
    plt.bar(x + 2 * width, y3 , color = "royalblue",width=width,label='MACPPO ')
    plt.xlabel("Number of vehicles")
    plt.ylabel("Total scores")
    plt.legend(loc = "best")
    plt.xticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9],['2','4','6','8','10','12','14','16','18','20'])
    # my_y_ticks = np.arange(0.8, 0.95, 0.02)
    # plt.ylim((0.8, 0.95))
    # plt.yticks(my_y_ticks)
    plt.savefig('total scores histogram.png', dpi=400)

def draw_percentage_improvement(x, y1, y2, y3):
    l0=plt.plot(x,np.zeros(len(x)),'black')
    l1=plt.plot(x,(y2-y1)/y1*100,'r--',label='MADQN vs random policy ')
    l2=plt.plot(x,(y3-y1)/y1*100,'g--',label='MACPPO vs random policy ')
    l3=plt.plot(x,(y3-y2)/y2*100,'b--',label='MACPPO vs MADQN ')
    plt.xticks([2, 4, 6, 8, 10, 12, 14, 16, 18, 20],['2','4','6','8','10','12','14','16','18','20'])
    plt.xlabel('Number of vehicles')
    plt.ylabel('Percentage improvement %')
    plt.legend()
    plt.savefig('percentage improvement.png', dpi=400)

def draw_2percentage_improvement(x, y1, y2, y3, y4, y5, y6):
    l0=plt.plot(x,np.zeros(len(x)),'black')
    l1=plt.plot(x,(y2-y1)/y1*100,'r--',label='MADQN vs random policy in 20 * 20 nodes ')
    l2=plt.plot(x,(y3-y1)/y1*100,'g--',label='MACPPO vs random policy in 20 * 20 nodes ')
    l3=plt.plot(x,(y3-y2)/y2*100,'b--',label='MACPPO vs MADQN in 20 * 20 nodes ')
    l4=plt.plot(x,(y5-y4)/y4*100,label='MADQN vs random policy in 30 * 30 nodes ')
    l5=plt.plot(x,(y6-y4)/y4*100,label='MACPPO vs random policy in 30 * 30 nodes ')
    l6=plt.plot(x,(y6-y5)/y5*100,label='MACPPO vs MADQN in 30 * 30 nodes ')
    plt.xticks([2, 4, 6, 8, 10, 12, 14, 16, 18, 20],['2','4','6','8','10','12','14','16','18','20'])
    plt.xlabel('Number of vehicles')
    plt.ylabel('Percentage improvement %')
    plt.legend()
    plt.savefig('percentage improvement.png', dpi=400)
    
def process_data_for_timestep_score(file_name):
    '''
    If you want to average the results of vehicles, remember to divide by the number of vehicles.
    '''
    with open(project_path + '/experiment_results/bug_in_env(episode_grid_score=episode_grid_score+episode_got_score)/{}.pickle'.format(file_name), 'rb') as file:
        data = pickle.load(file)
    mean_ts_score = np.mean(np.array(data[data['best_state']['test_session']]['all_timesteps_socre']), axis=0)
    return mean_ts_score  # / vehicle_number

def not_cumulative_score(mean_ts_score):
    all = 0
    for i in range(len(mean_ts_score)):
        new_all = mean_ts_score[i]
        mean_ts_score[i] -= all
        all = new_all
    return mean_ts_score

def draw_timestep_score(y1, y2, y3, y1r, y2r, y3r, score_type):
    fontsize = 12
    plt.figure(figsize=(8, 6))
    l1r=plt.plot(range(len(y1r)),y1r,'r',label='2 vehicles random policy ')
    l2r=plt.plot(range(len(y2r)),y2r,'g',label='4 vehicles random policy ')
    l3r=plt.plot(range(len(y3r)),y3r,'b',label='6 vehicles random policy ')
    l1=plt.plot(range(len(y1)),y1,'r--',label='2 vehicles MACPPO ')
    l2=plt.plot(range(len(y2)),y2,'g--',label='4 vehicles MACPPO ')
    l3=plt.plot(range(len(y3)),y3,'b--',label='6 vehicles MACPPO ')
    plt.tick_params(labelsize=fontsize)
    plt.xticks(range(len(y1)),np.array(range(1, len(y1)+1))*3)
    plt.xlabel('Minutes', size=fontsize)
    if score_type == 'cumulative':
        ylabel = 'Average vehicle cumulative scores'
        save_name = 'cumulative_timestep_score.png'
    else:
        ylabel = 'Average vehicle scores'
        save_name = 'timestep_score.png'
    plt.ylabel(ylabel, size=fontsize)
    plt.legend(fontsize=fontsize)
    plt.savefig(save_name, dpi=400)

In [3]:
x = np.array([2, 4, 6, 8, 10, 12, 14, 16, 18, 20])
y_20_random = np.array([0.1518, 0.2758, 0.3867, 0.4762, 0.5474, 0.6113, 0.6723, 0.7197, 0.7625, 0.7920])
y_20_DQN = np.array([0.1851, 0.3178, 0.4286, 0.5101, 0.5784, 0.6274, 0.6850, 0.7252, 0.7677, 0.8037])
y_20_PPO = np.array([0.2232, 0.3320, 0.4358, 0.5257, 0.5923, 0.6583, 0.6972, 0.7513, 0.7876, 0.8073])
y_30_random = np.array([0.0660, 0.1311, 0.1942, 0.2387, 0.2925, 0.3382, 0.3809, 0.4273, 0.4673, 0.5000])
y_30_DQN = np.array([0.0970, 0.1788, 0.2470, 0.3169, 0.3610, 0.4145, 0.4636, 0.5071, 0.5348, 0.5650])
y_30_PPO = np.array([0.1030, 0.1920, 0.2505, 0.3111, 0.3617, 0.4076, 0.4463, 0.4961, 0.5435, 0.5690])

In [4]:
print('Algorithm improvement relative to random policy')
print('****************************************************************************************************************')
print('env20_DQN_vs_random: ', list((y_20_DQN-y_20_random)/y_20_random*100))
print('****************************************************************************************************************')
print('env20_PPO_vs_random: ', list((y_20_PPO-y_20_random)/y_20_random*100))
print('****************************************************************************************************************')
print('env30_DQN_vs_random: ', list((y_30_DQN-y_30_random)/y_30_random*100))
print('****************************************************************************************************************')
print('env30_PPO_vs_random: ', list((y_30_PPO-y_30_random)/y_30_random*100))

Algorithm improvement relative to random policy
****************************************************************************************************************
env20_DQN_vs_random:  [21.93675889328063, 15.2284263959391, 10.835272821308505, 7.118857622847539, 5.663134819145054, 2.6337313921151653, 1.8890376320095263, 0.7642073086007988, 0.6819672131147664, 1.4772727272727186]
****************************************************************************************************************
env20_PPO_vs_random:  [47.03557312252966, 20.377084844089932, 12.697181277476089, 10.394792104157903, 8.202411399342354, 7.688532635367258, 3.703703703703708, 4.39071835487008, 3.2918032786885263, 1.9318181818181794]
****************************************************************************************************************
env30_DQN_vs_random:  [46.96969696969697, 36.38443935926773, 27.18846549948506, 32.760787599497284, 23.41880341880342, 22.560615020697806, 21.711735363612494, 18.675403697636316,

In [5]:
# draw_histogram(y_20_random, y_20_DQN, y_20_PPO)

In [6]:
# draw_percentage_improvement(x, y_30_random, y_30_DQN, y_30_PPO)

In [7]:
# draw_2percentage_improvement(x, y_20_random, y_20_DQN, y_20_PPO, y_30_random, y_30_DQN, y_30_PPO)

In [8]:
# score_type = 'cumulative'
# y1r = np.array([0.02569412561404415, 0.03972090221120692, 0.048846038884742295, 0.056639406103038255, 0.0649540524546637, 0.0730204958644329, 0.0811953358829186, 0.08719998195489141, 0.09366132150805156, 0.10023539966112001, 0.10642002293570871, 0.1130171385755208, 0.11945023570963946, 0.12551441874889308, 0.12870562173267613, 0.13433544886267354, 0.14025279767523113, 0.1446772833214185, 0.15055814557367883, 0.15577575950816147])
# y1r /= 2
# y2r = np.array([0.051693315556860304, 0.07562942879285663, 0.09533336679361773, 0.11054037832739105, 0.12531218942989705, 0.13724578049153682, 0.15044459414947156, 0.16348685286954276, 0.17753681259889362, 0.18886665608476355, 0.19862234928788383, 0.20715089932903968, 0.21752436930347208, 0.2293713637943599, 0.23664084135225186, 0.2458020989870578, 0.25496321762676916, 0.2641648596250075, 0.27252417982243426, 0.27977274431820887])
# y2r /= 4
# y3r = np.array([0.07774972965263523, 0.11133320865942385, 0.13850253429530635, 0.15764828969495118, 0.17772200984122782, 0.1988249065306409, 0.2170988532469856, 0.23306985012165993, 0.2481019194403476, 0.263339469352903, 0.28039233669302766, 0.29619935294634187, 0.3086813376713476, 0.32180983796570445, 0.3320382820109385, 0.3437048088904832, 0.3533077326492773, 0.36382591322512636, 0.372656216260404, 0.38249211110528736])
# y3r /= 6

# y1 = process_data_for_timestep_score('PPO_state_vehicle2_env_20_20') / 2
# y2 = process_data_for_timestep_score('PPO_state_vehicle4_env_20_20') / 4
# y3 = process_data_for_timestep_score('PPO_state_vehicle6_env_20_20') / 6

# for mean_ts_score in [y1r, y2r, y3r, y1, y2, y3]:
#     mean_ts_score = not_cumulative_score(mean_ts_score)
# score_type = 'not_cumulative'

In [9]:
# draw_timestep_score(y1, y2, y3, y1r, y2r, y3r, score_type)

In [10]:
vn = 6
env = 20

In [19]:
try:   
    with open(project_path + '/experiment_results/bug_in_env(episode_grid_score=episode_grid_score+episode_got_score)/PPO_state_vehicle{}_env_{}_{}.pickle'.format(vn, env, env), 'rb') as file:
        a = pickle.load(file)
    print(a.keys())
except:
    pass

dict_keys([1, 'best_state', 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217

In [20]:
a['best_state'].keys()

dict_keys(['test_100_episodes_mean_total_score', 'policy_params', 'test_session', 'best_episode', 'best_train_session', 'random_policy_100_episodes_mean_total_score', 'the_best_100_episodes_mean_total_score'])

In [21]:
a['best_state']['policy_params']

{0: OrderedDict([('extractor.policy_conv_net.0.weight',
               tensor([[[ 4.9603e-02, -9.0680e-02, -1.1755e-01,  7.4903e-02,  2.0908e-01],
                        [-1.1091e-01, -1.1822e-01, -1.0510e-01,  6.5718e-02, -2.1469e-01],
                        [-1.8614e-01,  7.8496e-03, -2.8049e-01, -2.5633e-01, -3.2870e-02]],
               
                       [[-4.3512e-02,  1.3327e-01, -7.1308e-02, -7.5844e-02, -1.7340e-01],
                        [ 1.4298e-01, -2.3898e-01,  5.8205e-02, -2.2469e-01, -1.5648e-01],
                        [-4.1214e-03,  2.7187e-01,  6.1539e-03,  2.3077e-01, -2.1990e-01]],
               
                       [[-1.9034e-01, -2.2477e-01,  1.1756e-01, -1.0330e-03, -2.4474e-01],
                        [-7.6123e-02, -1.9440e-01, -2.0076e-01, -1.1890e-01, -6.1740e-02],
                        [ 2.3447e-01,  1.0540e-01,  7.9360e-05,  1.3437e-01, -1.3009e-01]],
               
                       [[ 1.0673e-01,  7.2695e-02, -2.5031e-01,  1.6398e-0

In [12]:
# try:
#     with open(project_path + '/experiment_results/bug_in_env/PPO_state_vehicle{}_env_{}_{}.pickle'.format(vn, env, env), 'rb') as file:
#         a = pickle.load(file)
#     print('**************************************PPO**************************************')
#     print('test_100_episodes_mean_total_score: ', a['best_state']['test_100_episodes_mean_total_score'])
#     print('the_best_100_episodes_mean_total_score: ', a['best_state']['the_best_100_episodes_mean_total_score'])
#     print('*******************************************************************************')
# except:
#     pass

In [13]:
# try:
#     with open(project_path + '/experiment_results/bug_in_env/QL_state_vehicle{}_env_{}_{}.pickle'.format(vn, env, env), 'rb') as file:
#         a = pickle.load(file)
#     print('**************************************QL**************************************')
#     print('test_100_episodes_mean_total_score: ', a['best_state']['test_100_episodes_mean_total_score'])
#     print('the_best_100_episodes_mean_total_score: ', a['best_state']['the_best_100_episodes_mean_total_score'])
#     print('**************************************QL**************************************')
# except:
#     pass

In [28]:
# file_name = 'bug_in_env_episode_duration_5h'
# vehicle_num = 6
# height = 20
# width = 20
# episode_duration = 3600 * 5
# train_link_weight_distribution = 'UD'
# test_link_weight_distribution = 'UD'

# try:
#     with open(project_path + '/experiment_results/{}/PPO_state_vehicle{}_env_{}_{}_ed_{}_trainD_{}_testD_{}.pickle'.format(
#     file_name, vehicle_num, height, width, episode_duration, train_link_weight_distribution, test_link_weight_distribution
#     ), 'rb') as file:
#         data = pickle.load(file)
# except: 
#     with open(project_path + '/experiment_results/{}/PPO_state_vehicle{}_env_{}_{}.pickle'.format(
#     file_name, vehicle_num, height, width
#     ), 'rb') as file:
#         data = pickle.load(file)

# episodes_got_scores = np.array(data[data['best_state']['test_session']]['episodes_got_scores'])
# episodes_grid_scores = np.array(data[data['best_state']['test_session']]['episodes_grid_scores'])
# # episodes_grid_scores = episodes_grid_scores + episodes_got_scores

# mean_episodes_got_scores = np.mean(episodes_got_scores, axis=0)
# mean_episodes_grid_scores = np.mean(episodes_grid_scores, axis=0)
# (mean_episodes_grid_scores * np.log(mean_episodes_grid_scores / mean_episodes_got_scores)).sum()

0.20835324662774532

In [29]:
# with open(project_path + '/egs.pickle', 'rb') as file:
#     data = pickle.load(file)

# episodes_got_scores = np.array(data['episodes_got_scores'])
# episodes_grid_scores = np.array(data['episodes_grid_scores'])

# mean_episodes_got_scores = np.mean(episodes_got_scores, axis=0)
# mean_episodes_grid_scores = np.mean(episodes_grid_scores, axis=0)
# (mean_episodes_grid_scores * np.log(mean_episodes_grid_scores / mean_episodes_got_scores)).sum()

In [36]:
with open(project_path + '/random_policy_grid_score.pickle', 'rb') as file:
    data = pickle.load(file)

episodes_got_scores = np.array(data['episodes_got_scores'])
episodes_grid_scores = np.array(data['episodes_grid_scores'])

mean_episodes_got_scores = np.mean(episodes_got_scores, axis=0)
mean_episodes_grid_scores = np.mean(episodes_grid_scores, axis=0)
(mean_episodes_grid_scores * np.log(mean_episodes_grid_scores / mean_episodes_got_scores)).sum()

0.2955602080541953

In [37]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

COLORS = ['#00E400', '#FFFF00', '#FF7E00', '#FF0000', '#99004C', '#7E0023']
CMAP = mcolors.LinearSegmentedColormap.from_list("", COLORS, N=500)
NORM = mcolors.Normalize(vmax=0.0224, vmin=0)
# NORM = mcolors.Normalize(vmax=experienced_travel_time_grid.max(), vmin=experienced_travel_time_grid.min())


def draw_grid_score(episodes_scores: np.ndarray, file_name: str):

    im = plt.imshow(episodes_scores, cmap=CMAP, norm=NORM, interpolation='none', origin='lower')
    plt.colorbar(im)
    plt.xlabel('Rows')
    plt.ylabel('Columns')
    plt.savefig(file_name, transparent=True, dpi=400)
    plt.close('all')

In [38]:
draw_grid_score(mean_episodes_got_scores, 'got_score.png')
# draw_grid_score(mean_episodes_grid_scores, 'grid_score.png')